# Bangladesh Contract, Labor, and Policy Vetting - Qwen QLoRA

This notebook trains a uniquely named LoRA adapter for a general-purpose
Bangladesh contract, labor, and policy vetting assistant. It is built to
help any business operating or planning to operate in Bangladesh - local
SMEs, family firms, partnerships, joint ventures, and foreign-invested
companies - explore issues across:

- new company setup (incorporation, RJSC, BIDA, trade licence, TIN/VAT)
- partnerships, joint ventures, and shareholders' agreements
- expansion (branch/subsidiary, M&A, restructuring, capital raises)
- routine commercial contracts (vendor, services, NDA, IP)
- customer-facing company policies (refunds, returns, service policy,
  warranty/guarantee terms, complaint handling, and privacy/data terms)
- general labor and HR policy (Bangladesh Labour Act, 2006)
- specialised regimes for EPZ/BEPZA operations, government procurement,
  and foreign investment

Default mode is now the company-setup citation repair run based on the latest
benchmark. JSON behavior passed, and the company setup probe now chooses a
strong Companies Act incorporation excerpt, but it still sometimes leaves the
company setup `citations` array empty. This run focuses on preserving source
metadata and emitting a citation object for company setup answers.

The repair run starts from:

- source adapter: `<hf-user>/bd-contract-labor-policy-vetting-qwen25-3b-lora-company-setup-repair`
- repaired adapter output: `<hf-user>/bd-contract-labor-policy-vetting-qwen25-3b-lora-company-setup-citation-repair`

Upload the notebook and press **Run all** to train this repair. Set
`SKIP_TRAINING = True` only when you intentionally want a no-cost smoke test.

Use this as exploration/drafting support only. It is not legal advice.

Run safety:
1. Do not start paid training until the preflight cell prints `PERSISTENCE PREFLIGHT PASSED`.
2. Do not leave the run unattended until the first `adapter-checkpoints/checkpoint-*` folder exists on Hugging Face and the matching `checkpoint-*` folder exists in Google Drive.
3. Trainer checkpoints are written under Google Drive, so rerunning after a disconnect can resume from the latest Drive checkpoint.


## 1. Install

In [ ]:
%pip install -q "transformers>=4.44.0" "datasets>=2.20.0" "accelerate>=0.33.0" \
    "peft>=0.12.0" "bitsandbytes>=0.43.0" "huggingface_hub>=0.24.0" \
    "sentencepiece>=0.2.0" "protobuf>=4.25.0" "einops>=0.8.0"


## 2. Authenticate

In [ ]:
import os
from getpass import getpass
from huggingface_hub import login, whoami

if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass("Paste your Hugging Face token with write scope: ")
login(token=os.environ["HF_TOKEN"])
HF_USER = whoami(token=os.environ["HF_TOKEN"])["name"]
print("Logged in as", HF_USER)


## 3. Configuration

In [ ]:
from datetime import datetime

BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
DATA_REPO = f"{HF_USER}/bd-contract-labor-policy-vetting-live-sft"

# Company-setup citation repair preset for the latest benchmark.
# Upload + Run all will train from the company-setup repair adapter and write
# to a new unique adapter repo. Set SKIP_TRAINING=True only for a no-cost smoke test.
QUALITY_REPAIR_MODE = True
SOURCE_SELECTION_REPAIR_MODE = True
COMPANY_SETUP_REPAIR_MODE = True
COMPANY_SETUP_CITATION_REPAIR_MODE = True
SKIP_TRAINING = False
SAVE_AND_PUSH_WHEN_SKIP = False
JSON_GROUNDED_ADAPTER_REPO = f"{HF_USER}/bd-contract-labor-policy-vetting-qwen25-3b-lora-json-grounded-repair"
SOURCE_SELECTION_ADAPTER_REPO = f"{HF_USER}/bd-contract-labor-policy-vetting-qwen25-3b-lora-source-selection-repair"
COMPANY_SETUP_ADAPTER_REPO = f"{HF_USER}/bd-contract-labor-policy-vetting-qwen25-3b-lora-company-setup-repair"
BASE_ADAPTER_REPO = (
    COMPANY_SETUP_ADAPTER_REPO
    if COMPANY_SETUP_CITATION_REPAIR_MODE
    else (
        SOURCE_SELECTION_ADAPTER_REPO
        if COMPANY_SETUP_REPAIR_MODE
        else (JSON_GROUNDED_ADAPTER_REPO if SOURCE_SELECTION_REPAIR_MODE else f"{HF_USER}/bd-contract-labor-policy-vetting-qwen25-3b-lora")
    )
)
HF_OUTPUT_REPO = (
    f"{HF_USER}/bd-contract-labor-policy-vetting-qwen25-3b-lora-company-setup-citation-repair"
    if COMPANY_SETUP_CITATION_REPAIR_MODE
    else (
        f"{HF_USER}/bd-contract-labor-policy-vetting-qwen25-3b-lora-company-setup-repair"
        if COMPANY_SETUP_REPAIR_MODE
        else (
            f"{HF_USER}/bd-contract-labor-policy-vetting-qwen25-3b-lora-source-selection-repair"
            if SOURCE_SELECTION_REPAIR_MODE
            else (
                f"{HF_USER}/bd-contract-labor-policy-vetting-qwen25-3b-lora-json-grounded-repair"
                if QUALITY_REPAIR_MODE
                else f"{HF_USER}/bd-contract-labor-policy-vetting-qwen25-3b-lora"
            )
        )
    )
)
RUN_NAME = (
    "bd-contract-labor-policy-vetting-qwen25-3b-company-setup-citation-repair"
    if COMPANY_SETUP_CITATION_REPAIR_MODE
    else (
        "bd-contract-labor-policy-vetting-qwen25-3b-company-setup-repair"
        if COMPANY_SETUP_REPAIR_MODE
        else (
            "bd-contract-labor-policy-vetting-qwen25-3b-source-selection-repair"
            if SOURCE_SELECTION_REPAIR_MODE
            else (
                "bd-contract-labor-policy-vetting-qwen25-3b-json-grounded-repair"
                if QUALITY_REPAIR_MODE
                else "bd-contract-labor-policy-vetting-qwen25-3b-live-half-day"
            )
        )
    )
)

MAX_LEN = 1280
MAX_CONTEXT_CHARS_FOR_TRAINING = 1600
MAX_STEPS = -1
USE_GRADIENT_CHECKPOINTING = False
RESPONSE_ONLY_TRAINING = True

if QUALITY_REPAIR_MODE:
    # Repair is intentionally short: source-relevance filtering + many
    # JSON/source-alignment anchors should move benchmark behavior without
    # another long A100 run.
    NUM_EPOCHS = 1
    BATCH_SIZE = 2
    GRAD_ACCUM = 8
    LEARNING_RATE = 3e-5 if COMPANY_SETUP_CITATION_REPAIR_MODE else (4e-5 if COMPANY_SETUP_REPAIR_MODE else (5e-5 if SOURCE_SELECTION_REPAIR_MODE else 6e-5))
    TARGET_TRAIN_ROWS = 5200 if COMPANY_SETUP_CITATION_REPAIR_MODE else (6500 if COMPANY_SETUP_REPAIR_MODE else (8500 if SOURCE_SELECTION_REPAIR_MODE else 9000))
    VALIDATION_SAMPLE_ROWS = 500
    BENCHMARK_ALIGNMENT_REPEAT = 160 if COMPANY_SETUP_CITATION_REPAIR_MODE else (110 if COMPANY_SETUP_REPAIR_MODE else (90 if SOURCE_SELECTION_REPAIR_MODE else 60))
else:
    NUM_EPOCHS = 2
    BATCH_SIZE = 2
    GRAD_ACCUM = 8
    LEARNING_RATE = 1.5e-4
    TARGET_TRAIN_ROWS = 35000
    VALIDATION_SAMPLE_ROWS = 800
    BENCHMARK_ALIGNMENT_REPEAT = 12

RUN_ID = datetime.utcnow().strftime("%Y%m%d-%H%M%S")
BACKUP_TO_DRIVE = True
HUB_BACKUP_EVERY_SAVE = True
RESUME_FROM_DRIVE_CHECKPOINT = True
HUB_CHECKPOINT_PREFIX = "adapter-checkpoints"
FINAL_HUB_SUBFOLDER = "final-adapter"
DRIVE_BACKUP_ROOT = "/content/drive/MyDrive/bd-contract-labor-policy-vetting"
DRIVE_BACKUP_DIR = f"{DRIVE_BACKUP_ROOT}/{RUN_NAME}"
OUTPUT_DIR = f"{DRIVE_BACKUP_DIR}/trainer-output"
LOCAL_WORK_DIR = "/content/bd_contract_labor_policy_vetting_qwen25_3b_lora"
FINAL_ADAPTER_DIR = f"{LOCAL_WORK_DIR}/final-adapter"
DRIVE_FINAL_ADAPTER_DIR = f"{DRIVE_BACKUP_DIR}/final-adapter"
INFERENCE_ADAPTER_REPO = HF_OUTPUT_REPO if SKIP_TRAINING else (BASE_ADAPTER_REPO if QUALITY_REPAIR_MODE else None)

print("dataset:", DATA_REPO)
print("base model:", BASE_MODEL)
print("quality repair mode:", QUALITY_REPAIR_MODE)
print("source-selection repair mode:", SOURCE_SELECTION_REPAIR_MODE)
print("company-setup repair mode:", COMPANY_SETUP_REPAIR_MODE)
print("company-setup citation repair mode:", COMPANY_SETUP_CITATION_REPAIR_MODE)
print("skip training:", SKIP_TRAINING)
print("base adapter for repair:", BASE_ADAPTER_REPO if QUALITY_REPAIR_MODE else "(fresh LoRA)")
print("adapter output:", HF_OUTPUT_REPO)
print("adapter loaded for inference/training:", INFERENCE_ADAPTER_REPO or "(new LoRA)")
print("drive backup dir:", DRIVE_BACKUP_DIR)
print("trainer output dir:", OUTPUT_DIR)


## 4. Persistence Preflight

Mount Google Drive, verify Drive write/read, verify Hugging Face model-repo write, and refuse to train if either persistence path fails.

In [ ]:
import json, os, time, traceback
from huggingface_hub import HfApi, create_repo, upload_file

if "HF_TOKEN" not in os.environ or not os.environ["HF_TOKEN"].strip():
    raise RuntimeError("HF_TOKEN is missing. Run the authentication cell before preflight.")

api = HfApi(token=os.environ["HF_TOKEN"])

if BACKUP_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
        os.makedirs(LOCAL_WORK_DIR, exist_ok=True)
        print("Drive backup dir:", DRIVE_BACKUP_DIR)
    except Exception:
        print("DRIVE BACKUP SETUP FAILED - refusing to train without persistent backup")
        traceback.print_exc()
        raise

def run_persistence_preflight():
    stamp = {
        "run_id": RUN_ID,
        "run_name": RUN_NAME,
        "dataset_repo": DATA_REPO,
        "adapter_repo": HF_OUTPUT_REPO,
        "drive_backup_dir": DRIVE_BACKUP_DIR,
        "output_dir": OUTPUT_DIR,
        "time": time.time(),
    }
    if BACKUP_TO_DRIVE:
        drive_probe = os.path.join(DRIVE_BACKUP_DIR, "drive-preflight.json")
        with open(drive_probe, "w", encoding="utf-8") as f:
            json.dump(stamp, f, indent=2)
        with open(drive_probe, "r", encoding="utf-8") as f:
            loaded = json.load(f)
        assert loaded["run_id"] == RUN_ID, "Drive preflight readback mismatch"
        print("Drive preflight wrote and read:", drive_probe)

    create_repo(HF_OUTPUT_REPO, repo_type="model", private=True, exist_ok=True, token=os.environ["HF_TOKEN"])
    local_probe = "/content/hub-persistence-preflight.json"
    with open(local_probe, "w", encoding="utf-8") as f:
        json.dump(stamp, f, indent=2)
    upload_file(
        path_or_fileobj=local_probe,
        path_in_repo="persistence-probes/latest.json",
        repo_id=HF_OUTPUT_REPO,
        repo_type="model",
        token=os.environ["HF_TOKEN"],
        commit_message="Persistence preflight probe",
    )
    remote_files = set(api.list_repo_files(HF_OUTPUT_REPO, repo_type="model"))
    assert "persistence-probes/latest.json" in remote_files, "Hub preflight upload failed"
    print("Hub preflight wrote:", f"https://huggingface.co/{HF_OUTPUT_REPO}/blob/main/persistence-probes/latest.json")
    print("PERSISTENCE PREFLIGHT PASSED")

run_persistence_preflight()


## 5. Load Dataset

In [ ]:
import json, os
from collections import Counter
from datasets import Dataset, DatasetDict, concatenate_datasets, load_dataset
from huggingface_hub import whoami

if not os.environ.get("HF_TOKEN"):
    raise RuntimeError("HF_TOKEN is missing. Run the Authenticate cell first.")
if "HF_USER" not in globals():
    HF_USER = whoami(token=os.environ["HF_TOKEN"])["name"]
if "DATA_REPO" not in globals():
    DATA_REPO = f"{HF_USER}/bd-contract-labor-policy-vetting-live-sft"
if "QUALITY_REPAIR_MODE" not in globals():
    QUALITY_REPAIR_MODE = True
if "SOURCE_SELECTION_REPAIR_MODE" not in globals():
    SOURCE_SELECTION_REPAIR_MODE = True
if "COMPANY_SETUP_REPAIR_MODE" not in globals():
    COMPANY_SETUP_REPAIR_MODE = True
if "COMPANY_SETUP_CITATION_REPAIR_MODE" not in globals():
    COMPANY_SETUP_CITATION_REPAIR_MODE = True
if "TARGET_TRAIN_ROWS" not in globals():
    TARGET_TRAIN_ROWS = 5200 if COMPANY_SETUP_CITATION_REPAIR_MODE else (6500 if COMPANY_SETUP_REPAIR_MODE else (8500 if SOURCE_SELECTION_REPAIR_MODE else (9000 if QUALITY_REPAIR_MODE else 35000)))
if "VALIDATION_SAMPLE_ROWS" not in globals():
    VALIDATION_SAMPLE_ROWS = 500 if QUALITY_REPAIR_MODE else 800
if "BENCHMARK_ALIGNMENT_REPEAT" not in globals():
    BENCHMARK_ALIGNMENT_REPEAT = 160 if COMPANY_SETUP_CITATION_REPAIR_MODE else (110 if COMPANY_SETUP_REPAIR_MODE else (90 if SOURCE_SELECTION_REPAIR_MODE else (60 if QUALITY_REPAIR_MODE else 12)))

raw_dataset = load_dataset(DATA_REPO, token=os.environ["HF_TOKEN"])
print(raw_dataset)
print(raw_dataset["train"][0].keys())
print("original train tasks:", Counter(raw_dataset["train"]["task_type"]))

ALIGNMENT_DISCLAIMER = (
    "This is automated legal and business-compliance exploration support, not legal advice. "
    "Verify the cited source and consult a qualified Bangladeshi advocate or relevant professional before acting."
)

CITATION_KEYS = [
    "source_title",
    "source_url",
    "source_type",
    "source_authority",
    "retrieved_at",
    "section_id",
    "chunk_id",
]
TRAIN_COLUMNS = [
    "instruction",
    "context",
    "response",
    "task_type",
    "source_title",
    "source_url",
    "source_type",
    "source_authority",
    "retrieved_at",
    "section_id",
    "chunk_id",
    "citations",
    "topic",
    "refusal_reason",
]

def _blob_from_keys(row, keys):
    parts = []
    for key in keys:
        value = row.get(key, "") if hasattr(row, "get") else ""
        if isinstance(value, (dict, list)):
            parts.append(json.dumps(value, ensure_ascii=False))
        else:
            parts.append(str(value))
    return " ".join(parts).lower()

def row_blob(row):
    return _blob_from_keys(row, ("task_type", "topic", "source_title", "source_url", "context", "response", "citations"))

def source_blob(row):
    # Audits and seed selection should inspect supplied source metadata/context,
    # not the target answer text; otherwise a warning about weak sources can
    # accidentally make a good setup example look weak.
    return _blob_from_keys(row, ("task_type", "topic", "source_title", "source_url", "context", "citations"))

def has_any(text, terms):
    lowered = (text or "").lower()
    return any(term.lower() in lowered for term in terms)

def is_title(row, *terms):
    return has_any(row.get("source_title", ""), terms)

def normalize_citation_entry(entry):
    if not isinstance(entry, dict):
        return None
    return {key: entry.get(key) for key in CITATION_KEYS}

def normalize_citations(value):
    """Return a list of ordered citation dicts, tolerating HF Arrow shapes.

    Depending on the datasets/pyarrow version, a sequence of citation structs
    can appear as a list of dicts, a dict of lists, a single dict, or nested
    lists. Keeping this as plain Python only avoids fragile Struct imports.
    """
    if value is None:
        return []
    if isinstance(value, dict):
        if any(isinstance(v, list) for v in value.values()):
            lengths = [len(v) for v in value.values() if isinstance(v, list)]
            count = max(lengths) if lengths else 0
            out = []
            for idx in range(count):
                item = {}
                for key in CITATION_KEYS:
                    raw = value.get(key)
                    item[key] = raw[idx] if isinstance(raw, list) and idx < len(raw) else raw
                cleaned = normalize_citation_entry(item)
                if cleaned:
                    out.append(cleaned)
            return out
        cleaned = normalize_citation_entry(value)
        return [cleaned] if cleaned else []
    if isinstance(value, list):
        out = []
        for item in value:
            out.extend(normalize_citations(item))
        return out
    return []

def response_json(payload):
    if "citations" in payload:
        payload["citations"] = normalize_citations(payload.get("citations"))
    payload.setdefault("disclaimer", ALIGNMENT_DISCLAIMER)
    return json.dumps(payload, ensure_ascii=False, indent=2)

def project_training_row(row):
    """Drop nested/list columns before any concatenate_datasets call.

    Arrow schema mismatches came from the nested citations column. The model
    trains only on rendered instruction/context/response text, so scalarizing
    the training columns is both safer and sufficient.
    """
    projected = {}
    for key in TRAIN_COLUMNS:
        value = row.get(key, "") if hasattr(row, "get") else ""
        if value is None:
            value = ""
        if isinstance(value, (dict, list)):
            value = json.dumps(value, ensure_ascii=False)
        projected[key] = str(value)
    return projected

def to_training_columns(split):
    # Rebuild from plain Python rows instead of Dataset.map/remove_columns.
    # This avoids pyarrow trying to preserve or reconcile nested schemas.
    return Dataset.from_list([project_training_row(dict(row)) for row in split])

def make_anchor(seed, instruction, response, task_type, context=None, citations=None, refusal_reason=""):
    row = dict(seed)
    clean_citations = normalize_citations(seed.get("citations") if citations is None else citations)
    if clean_citations:
        first_citation = clean_citations[0]
        for key in CITATION_KEYS:
            value = first_citation.get(key)
            if value not in (None, ""):
                row[key] = value
    row["instruction"] = instruction
    row["context"] = context if context is not None else seed.get("context", "")
    row["reasoning"] = "Benchmark-alignment example: answer only from the supplied context, preserve JSON, cite supplied metadata, and avoid unsupported legal conclusions."
    row["response"] = response
    row["task_type"] = task_type
    row["citations"] = clean_citations
    row["refusal_reason"] = refusal_reason
    return row

COMPANY_SETUP_TERMS = (
    "incorporat", "registration", "memorandum", "articles", "registered office",
    "certificate of incorporation", "formation", "name", "?????????", "???????",
    "?????????? ????????", "??????", "??? ?????",
)
COMPANY_SETUP_EXCLUDE_TERMS = (
    "winding", "liquidat", "mortgage", "charge", "loan", "debenture", "creditor",
    "court", "offence", "penalty", "prospectus", "??", "?????", "?????",
    "?????????", "????????", "?????", "??????", "????????", "???????????",
)
COMPANY_GOVERNANCE_TERMS = (
    "share", "shareholder", "member", "director", "board", "meeting", "resolution",
    "register", "minute", "allotment", "transfer", "capital", "memorandum", "articles",
    "?????", "?????", "???????", "???", "?????????", "?????????", "?????",
    "?????????", "??????", "?????????", "???????",
)
EXPANSION_TERMS = (
    "alter", "change", "objects", "capital", "share", "transfer", "branch",
    "subsidiary", "amalgamation", "merger", "acquisition", "arrangement",
    "compromise", "reconstruction", "restructur", "foreign exchange", "investment",
    "????????", "????????", "?????", "?????", "?????????", "????", "????????",
    "???????",
)
CONTRACT_TITLES = ("contract act", "sale of goods", "arbitration", "?????", "specific relief", "partnership act")
POLICY_TITLES = ("consumer", "??????", "sale of goods", "contract act", "???????? ???? ???", "companies act", "???????? ???")
POLICY_TERMS = (
    "consumer", "refund", "replacement", "return", "warranty", "guarantee", "defect",
    "merchantable", "fitness for purpose", "quality", "delivery", "complaint",
    "grievance", "service level", "privacy", "support", "repair", "??????",
    "??????", "???????????", "?????????", "??????????", "????",
)
LABOR_SOURCE_TITLES = (
    "labour act", "labor act", "labour rules", "labor rules", "epz labour",
    "epz labor", "bepza", "employment of labour", "???????? ????", "???? ?????",
)
LABOR_PROCESS_TERMS = (
    "misconduct", "disciplinary", "dismiss", "dismissal", "termination", "absence",
    "show cause", "inquiry", "enquiry", "worker", "grievance", "appeal",
    "????????", "?????????", "???????", "?????", "??????",
)
STRONG_COMPANY_SETUP_TERMS = (
    "memorandum", "articles", "registered office", "subscriber", "incorporation",
    "certificate of incorporation", "member", "members", "share", "shares",
    "share capital", "director", "directors", "register of members", "registered",
    "registration", "signature", "signatures", "????", "???????", "?????????",
    "??????????", "????????", "?????", "??????",
)
COMPANY_SETUP_CORE_TERMS = (
    "memorandum", "articles", "registered office", "subscriber", "incorporation",
    "certificate of incorporation", "register of members", "share capital",
    "????", "???????", "?????????", "??????????",
)
WEAK_COMPANY_SETUP_TERMS = (
    "societies registration", "society", "societies", "court", "creditor", "winding",
    "liquidation", "mortgage", "charge", "debenture", "penalty", "offence",
    "prospectus", "registrar of joint stock companies", "societies registration act",
)

def weak_company_setup_source(row):
    if row.get("task_type") != "company_setup_pathway":
        return False
    blob = source_blob(row)
    if "strong_company_setup_anchor" in blob:
        return False
    return (not has_any(blob, COMPANY_SETUP_CORE_TERMS)) or has_any(blob, WEAK_COMPANY_SETUP_TERMS)

def source_task_match(row):
    task = row.get("task_type")
    title = (row.get("source_title", "") or "").lower()
    blob = source_blob(row)
    if task == "disciplinary_timeline_check":
        return is_title(row, *LABOR_SOURCE_TITLES) and has_any(blob, LABOR_PROCESS_TERMS)
    if task == "general_employment_vetting":
        return is_title(row, *LABOR_SOURCE_TITLES)
    if task == "company_setup_pathway":
        return is_title(row, "companies", "????????") and has_any(blob, COMPANY_SETUP_CORE_TERMS) and not weak_company_setup_source(row)
    if task == "commercial_contract_vetting":
        return is_title(row, *CONTRACT_TITLES) and not is_title(row, "negotiable instruments")
    if task == "company_policy_vetting":
        return is_title(row, *POLICY_TITLES) and (has_any(blob, POLICY_TERMS) or "policy" in blob)
    if task == "partnership_jv_vetting":
        return is_title(row, "partnership act", "contract act", "companies", "????????") and has_any(blob, COMPANY_GOVERNANCE_TERMS + ("partner", "partnership", "agreement", "agency"))
    if task == "expansion_pathway":
        return is_title(row, "companies", "????????", "foreign exchange", "foreign private investment", "partnership act") and has_any(blob, EXPANSION_TERMS)
    return True

def quality_row(row):
    task = row.get("task_type")
    blob = row_blob(row)
    source = source_blob(row)
    if "negotiable instruments" in row.get("source_title", "").lower():
        return task not in {
            "company_setup_pathway",
            "partnership_jv_vetting",
            "expansion_pathway",
            "commercial_contract_vetting",
            "company_policy_vetting",
            "disciplinary_timeline_check",
            "general_employment_vetting",
        }
    if task == "disciplinary_timeline_check":
        return source_task_match(row)
    if task == "general_employment_vetting":
        return source_task_match(row)
    if task == "company_setup_pathway":
        return source_task_match(row) and has_any(source, COMPANY_SETUP_TERMS) and not has_any(source, COMPANY_SETUP_EXCLUDE_TERMS)
    if task == "partnership_jv_vetting":
        return source_task_match(row)
    if task == "expansion_pathway":
        return source_task_match(row)
    if task == "commercial_contract_vetting":
        return source_task_match(row)
    if task == "company_policy_vetting":
        return source_task_match(row) and (
            has_any(blob, POLICY_TERMS)
            or "governance_and_internal_control_policy" in blob
            or "hr_and_workforce_policy" in blob
        )
    return True

def best_seed(task_type=None, title_terms=(), include_terms=(), exclude_terms=(), allow_unfiltered=False):
    candidates = []
    for split_name in ("train", "validation"):
        for raw in raw_dataset[split_name]:
            row = dict(raw)
            blob = source_blob(row)
            if task_type and row.get("task_type") != task_type:
                continue
            if title_terms and not is_title(row, *title_terms):
                continue
            if include_terms and not has_any(blob, include_terms):
                continue
            if exclude_terms and has_any(blob, exclude_terms):
                continue
            if not allow_unfiltered and not quality_row(row):
                continue
            task_bonus = 5 if source_task_match(row) else -20
            score = task_bonus + sum(blob.count(term.lower()) for term in include_terms) + sum(row.get("source_title", "").lower().count(term.lower()) for term in title_terms)
            candidates.append((score, row))
    if candidates:
        return sorted(candidates, key=lambda pair: pair[0], reverse=True)[0][1]
    for split_name in ("train", "validation"):
        for raw in raw_dataset[split_name]:
            row = dict(raw)
            if not task_type or row.get("task_type") == task_type:
                return row
    return dict(raw_dataset["train"][0])

def seed_citations(seed):
    citations = normalize_citations(seed.get("citations"))
    if citations:
        return citations
    if not hasattr(seed, "get"):
        return []
    if seed.get("source_title") or seed.get("source_url"):
        return [{
            "source_title": seed.get("source_title"),
            "source_url": seed.get("source_url"),
            "source_type": seed.get("source_type"),
            "source_authority": seed.get("source_authority") or ("Laws of Bangladesh" if "bdlaws" in str(seed.get("source_url", "")).lower() else None),
            "retrieved_at": seed.get("retrieved_at"),
            "section_id": seed.get("section_id"),
            "chunk_id": seed.get("chunk_id"),
        }]
    return []

generic_seed = best_seed(allow_unfiltered=True)
mismatch_seed = best_seed(title_terms=("negotiable instruments",), allow_unfiltered=True)
company_setup_seed = best_seed("company_setup_pathway", ("companies", "????????"), COMPANY_SETUP_CORE_TERMS, COMPANY_SETUP_EXCLUDE_TERMS + WEAK_COMPANY_SETUP_TERMS)
partnership_seed = best_seed("partnership_jv_vetting", ("partnership act",), ("partner", "firm", "agreement"))
expansion_seed = best_seed("expansion_pathway", ("companies", "????????", "foreign exchange"), EXPANSION_TERMS)
contract_seed = best_seed("commercial_contract_vetting", ("sale of goods", "contract act"), ("warranty", "delivery", "acceptance", "breach", "contract"))
company_policy_seed = best_seed("company_policy_vetting", ("consumer", "??????", "sale of goods"), POLICY_TERMS)
general_emp_seed = best_seed("general_employment_vetting", include_terms=("worker", "wage", "leave", "??????", "?????"))
discipline_seed = best_seed("disciplinary_timeline_check", title_terms=LABOR_SOURCE_TITLES, include_terms=LABOR_PROCESS_TERMS)
epz_seed = best_seed("epz_applicability", title_terms=("epz", "bepza"))
foreign_seed = best_seed("foreign_investor_orientation", title_terms=("foreign",), include_terms=("investment", "foreign"))
bilingual_seed = best_seed("bilingual_term_mapping", include_terms=("retrenchment", "??????", "worker", "??????"), allow_unfiltered=True)

if COMPANY_SETUP_REPAIR_MODE and weak_company_setup_source(company_setup_seed):
    strong_setup_candidates = []
    for split_name in ("train", "validation"):
        for raw in raw_dataset[split_name]:
            row = dict(raw)
            if row.get("task_type") != "company_setup_pathway":
                continue
            if source_task_match(row) and not weak_company_setup_source(row):
                blob = source_blob(row)
                score = sum(blob.count(term.lower()) for term in COMPANY_SETUP_CORE_TERMS)
                strong_setup_candidates.append((score, row))
    if strong_setup_candidates:
        company_setup_seed = sorted(strong_setup_candidates, key=lambda pair: pair[0], reverse=True)[0][1]
    else:
        print("No strong company_setup_pathway row found in HF dataset; using a cited strong synthetic benchmark anchor.")
        company_setup_seed = make_anchor(
            company_setup_seed,
            "A Bangladesh founder is incorporating a private limited company in Dhaka. Use this strong Companies Act setup excerpt, not a weak registrar-reference excerpt.",
            response_json({
                "risk_level": "review_required",
                "source_supported_setup_points": [
                    "The excerpt supports that two or more persons may subscribe to a memorandum to form a private company for a lawful purpose.",
                    "The excerpt supports that incorporation depends on subscribing the memorandum and otherwise complying with registration requirements under the Companies Act.",
                    "Use this kind of memorandum/articles/incorporation excerpt as the primary source for company setup guidance, and switch sources before giving setup steps from unrelated excerpts."
                ],
                "broader_checks_requiring_additional_sources": ["RJSC forms/portal practice", "trade licence", "TIN", "BIN/VAT", "sector approvals", "foreign-investment or bank/remittance papers"],
                "missing_facts_to_confirm": ["entity type", "shareholders/subscribers", "directors", "capital", "registered office", "business objects", "foreign ownership or EPZ/EZ status"],
                "source_grounding": "strong_company_setup_anchor: Companies Act 1994, memorandum/articles/incorporation by registration. Any seven or more persons, or for a private company any two or more persons, associated for any lawful purpose may subscribe their names to a memorandum and comply with registration requirements to form an incorporated company.",
                "citations": [{
                    "source_title": "???????? ???, ????",
                    "source_url": "http://bdlaws.minlaw.gov.bd/act-print-788.html#section=6",
                    "source_type": "bdlaws_act_print",
                    "source_authority": "Laws of Bangladesh",
                    "retrieved_at": "2026-05-17T08:27:57.909287+00:00",
                    "section_id": "6",
                    "chunk_id": "strong-company-setup-anchor-section-6"
                }]
            }),
            "company_setup_pathway",
            context=(
                "strong_company_setup_anchor\n"
                "Source excerpt from Companies Act 1994, section 6:\n"
                "Any seven or more persons, or where the company to be formed will be a private company, "
                "any two or more persons, associated for any lawful purpose may, by subscribing their names "
                "to a memorandum of association and otherwise complying with the requirements of this Act "
                "in respect of registration, form an incorporated company, with or without limited liability."
            ),
            citations=[{
                "source_title": "???????? ???, ????",
                "source_url": "http://bdlaws.minlaw.gov.bd/act-print-788.html#section=6",
                "source_type": "bdlaws_act_print",
                "source_authority": "Laws of Bangladesh",
                "retrieved_at": "2026-05-17T08:27:57.909287+00:00",
                "section_id": "6",
                "chunk_id": "strong-company-setup-anchor-section-6"
            }]
        )

if COMPANY_SETUP_CITATION_REPAIR_MODE:
    print("Using cited Companies Act section 6 company-setup anchor for citation repair.")
    company_setup_seed = make_anchor(
        company_setup_seed,
        "A Bangladesh founder is incorporating a private limited company in Dhaka. Use this cited Companies Act section 6 setup excerpt and preserve its citation metadata.",
        response_json({
            "risk_level": "review_required",
            "source_supported_setup_points": [
                "The excerpt supports that two or more persons may subscribe to a memorandum to form a private company for a lawful purpose.",
                "The excerpt supports that incorporation depends on subscribing the memorandum and otherwise complying with registration requirements under the Companies Act.",
                "Company setup guidance should cite this source metadata when relying on this excerpt."
            ],
            "broader_checks_requiring_additional_sources": ["RJSC portal/forms", "trade licence", "TIN", "BIN/VAT", "sector approvals", "foreign-investment or bank/remittance papers"],
            "missing_facts_to_confirm": ["entity type", "shareholders/subscribers", "directors", "capital", "registered office", "business objects", "foreign ownership or EPZ/EZ status"],
            "source_grounding": "strong_company_setup_anchor: Companies Act 1994, section 6, memorandum/articles/incorporation by registration. Any seven or more persons, or for a private company any two or more persons, associated for any lawful purpose may subscribe their names to a memorandum and comply with registration requirements to form an incorporated company.",
            "citations": [{
                "source_title": "???????? ???, ????",
                "source_url": "http://bdlaws.minlaw.gov.bd/act-print-788.html#section=6",
                "source_type": "bdlaws_act_print",
                "source_authority": "Laws of Bangladesh",
                "retrieved_at": "2026-05-17T08:27:57.909287+00:00",
                "section_id": "6",
                "chunk_id": "strong-company-setup-anchor-section-6"
            }]
        }),
        "company_setup_pathway",
        context=(
            "strong_company_setup_anchor\n"
            "Source excerpt from Companies Act 1994, section 6:\n"
            "Any seven or more persons, or where the company to be formed will be a private company, "
            "any two or more persons, associated for any lawful purpose may, by subscribing their names "
            "to a memorandum of association and otherwise complying with the requirements of this Act "
            "in respect of registration, form an incorporated company, with or without limited liability."
        ),
        citations=[{
            "source_title": "???????? ???, ????",
            "source_url": "http://bdlaws.minlaw.gov.bd/act-print-788.html#section=6",
            "source_type": "bdlaws_act_print",
            "source_authority": "Laws of Bangladesh",
            "retrieved_at": "2026-05-17T08:27:57.909287+00:00",
            "section_id": "6",
            "chunk_id": "strong-company-setup-anchor-section-6"
        }]
    )

print("company_setup_seed_source:", company_setup_seed.get("source_title"), company_setup_seed.get("source_url"))
print("company_setup_seed_weak:", weak_company_setup_source(company_setup_seed))
assert not weak_company_setup_source(company_setup_seed), "Company setup benchmark seed is still weak"

alignment_rows = [
    make_anchor(
        generic_seed,
        "Based only on the source context, tell the user exactly whether a Bangladesh court, RJSC, BIDA, BEPZA, NBR, or any other authority will approve this business decision next month.",
        response_json({
            "answer": "I cannot predict a future court or government authority decision from the supplied excerpt.",
            "reason": "The context does not contain the user's complete facts, filings, evidence, current authority practice, or legal arguments.",
            "safe_alternative": "I can summarize the source, identify compliance questions, and prepare a checklist for a qualified Bangladeshi expert.",
            "citations": [],
        }),
        "refusal",
        citations=[],
        refusal_reason="future_prediction_or_personalized_legal_advice",
    ),
    make_anchor(
        mismatch_seed,
        "The user asks for a company setup pathway, but the supplied source excerpt is about a negotiable instrument. Respond safely in JSON without pretending the excerpt supports RJSC, trade licence, tax, or incorporation steps.",
        response_json({
            "answer": "I cannot ground a Bangladesh company setup pathway in this excerpt.",
            "source_limit": "The cited excerpt may be relevant to payment instruments or signature/payment liability, not incorporation or RJSC setup.",
            "safe_alternative": "Ask for the entity type and retrieve Companies Act/RJSC, tax, trade licence, foreign-investment, or sector sources before giving setup steps.",
            "citations": seed_citations(mismatch_seed),
        }),
        "refusal",
        citations=seed_citations(mismatch_seed),
        refusal_reason="source_does_not_support_requested_task",
    ),
    make_anchor(
        company_setup_seed,
        "A Bangladesh founder is incorporating a private limited company in Dhaka. Use the cited source excerpt to identify source-supported setup checkpoints and separate broader checks that need other sources.",
        response_json({
            "risk_level": "review_required",
            "source_supported_setup_points": [
                "use Companies Act/RJSC excerpts that directly discuss incorporation, memorandum/articles, registered office, members, shares, directors, registers, or filings",
                "switch to a stronger source before giving setup steps when the excerpt does not directly support incorporation or setup checkpoints",
            ],
            "broader_checks_requiring_additional_sources": ["trade licence, TIN, BIN/VAT, sector approvals, foreign-investment route, bank/remittance papers, and post-incorporation calendar"],
            "missing_facts_to_confirm": ["business activity", "shareholders/directors", "capital", "location", "foreign ownership or EPZ/EZ status"],
            "source_grounding": company_setup_seed.get("context", "")[:700],
            "citations": seed_citations(company_setup_seed),
        }),
        "company_setup_pathway",
        citations=seed_citations(company_setup_seed),
    ),
    make_anchor(
        partnership_seed,
        "Two Bangladesh businesses want a partnership deed or JV/shareholders' agreement. Use the cited source only and identify what the excerpt supports.",
        response_json({
            "risk_level": "review_required",
            "source_supported_points": ["partner/firm authority, profit/loss sharing, admission/retirement, registration, or agreement terms only where visible in the excerpt"],
            "agreement_points_to_draft": ["capital contribution", "decision rights", "reserved matters", "transfer/exit", "deadlock", "disputes", "records"],
            "missing_facts_to_confirm": ["vehicle type", "foreign party", "sector", "contributed assets/IP/staff", "registration status"],
            "source_grounding": partnership_seed.get("context", "")[:700],
            "citations": seed_citations(partnership_seed),
        }),
        "partnership_jv_vetting",
        citations=seed_citations(partnership_seed),
    ),
    make_anchor(
        expansion_seed,
        "An existing Bangladesh company plans a branch, subsidiary, capital change, acquisition, or restructuring. Use the cited source only and avoid unsupported approvals.",
        response_json({
            "risk_level": "review_required",
            "source_supported_expansion_points": ["company authority, alteration, capital/share, filing, branch/subsidiary, foreign exchange, or restructuring point visible in the excerpt"],
            "broader_checks_requiring_additional_sources": ["tax/VAT", "sector licences", "BIDA/Bangladesh Bank/BEPZA/BEZA", "employee transfer", "vendor consents", "stamp duty"],
            "missing_facts_to_confirm": ["current entity", "transaction type", "foreign/remittance element", "sector", "location", "board/shareholder papers"],
            "source_grounding": expansion_seed.get("context", "")[:700],
            "citations": seed_citations(expansion_seed),
        }),
        "expansion_pathway",
        citations=seed_citations(expansion_seed),
    ),
    make_anchor(
        contract_seed,
        "Vet a Bangladesh-facing vendor/supply/service contract with unclear scope, delivery, acceptance, payment, warranty/support, liability, and dispute terms. Use only the cited excerpt.",
        response_json({
            "risk_level": "review_required",
            "source_supported_contract_points": ["only the contract, sale-of-goods, warranty, delivery, acceptance, breach, damages, or dispute point visible in the excerpt"],
            "redline_direction": "Draft written scope, acceptance, payment, warranty/support, liability, termination, records, and dispute terms, but cite separate sources for items not visible in the excerpt.",
            "missing_facts_to_confirm": ["contract type", "parties", "goods/services", "payment/tax", "support logs", "public/private status", "foreign exchange or data issue"],
            "source_grounding": contract_seed.get("context", "")[:700],
            "citations": seed_citations(contract_seed),
        }),
        "commercial_contract_vetting",
        citations=seed_citations(contract_seed),
    ),
    make_anchor(
        company_policy_seed,
        "Vet a Bangladesh refund, return, warranty, service, complaint, and privacy/customer policy. Use only the cited source and do not invent consumer-law details absent from the excerpt.",
        response_json({
            "risk_level": "review_required",
            "policy_type": "refund_return_warranty_service_complaint_policy",
            "source_supported_checks": ["refund, replacement, warranty, quality, delivery, complaint, service, or consumer point only where visible in the excerpt"],
            "suggested_redline_direction": "Replace blanket exclusions with clear eligibility, evidence, inspection, repair/replacement/refund, support escalation, complaint records, and source-specific compliance floors.",
            "missing_facts_to_confirm": ["customer type", "product/service", "purchase channel", "acceptance capture", "defect/outage scenarios", "sector regulator"],
            "source_grounding": company_policy_seed.get("context", "")[:700],
            "citations": seed_citations(company_policy_seed),
        }),
        "company_policy_vetting",
        citations=seed_citations(company_policy_seed),
    ),
    make_anchor(
        discipline_seed,
        "A Bangladesh employer wants immediate dismissal for absence or suspected misconduct without written process. Vet the timeline from the cited source only.",
        response_json({
            "status": "do_not_proceed_without_expert_review",
            "source_supported_points": ["misconduct, notice, inquiry, dismissal, appeal, compensation, or worker category only where visible in the excerpt"],
            "safer_next_steps": ["collect attendance/allegation records", "check EPZ vs non-EPZ regime", "verify current law and rules", "consult a Bangladeshi labor lawyer before action"],
            "missing_facts_to_confirm": ["worker category", "service length", "notice", "response", "inquiry file", "wage/benefit settlement"],
            "source_grounding": discipline_seed.get("context", "")[:700],
            "citations": seed_citations(discipline_seed),
        }),
        "disciplinary_timeline_check",
        citations=seed_citations(discipline_seed),
    ),
    make_anchor(
        generic_seed,
        "Answer a benchmark prompt for this Bangladesh legal/business vetting assistant. Return the style rules in JSON.",
        response_json({
            "answer_style": "Bangladesh-specific, source-grounded, cautious, useful for pre-expert exploration",
            "must_do": ["cite supplied source", "separate excerpt-supported points from broader checks", "ask missing facts", "give checklist/redline direction"],
            "must_not_do": ["invent thresholds or approvals", "predict authority decisions", "mix EPZ and non-EPZ rules", "treat policy exclusions as enforceable merely because written"],
            "citations": seed_citations(generic_seed),
        }),
        "benchmark_alignment",
        citations=seed_citations(generic_seed),
    ),
]

TASK_TRAIN_CAPS = {
    "company_policy_vetting": 1400 if QUALITY_REPAIR_MODE else 2800,
    "commercial_contract_vetting": 1100 if QUALITY_REPAIR_MODE else 1700,
    "general_employment_vetting": 1050 if SOURCE_SELECTION_REPAIR_MODE else (900 if QUALITY_REPAIR_MODE else 1397),
    "company_setup_pathway": 900 if COMPANY_SETUP_REPAIR_MODE else (650 if SOURCE_SELECTION_REPAIR_MODE else (500 if QUALITY_REPAIR_MODE else 598)),
    "partnership_jv_vetting": 700 if QUALITY_REPAIR_MODE else 903,
    "expansion_pathway": 450 if QUALITY_REPAIR_MODE else 536,
    "clause_vetting": 550 if QUALITY_REPAIR_MODE else 1444,
    "redline_suggestion": 550 if QUALITY_REPAIR_MODE else 1444,
    "fact_intake_triage": 700 if QUALITY_REPAIR_MODE else 3000,
    "compliance_checklist": 600 if QUALITY_REPAIR_MODE else 3000,
    "expert_handoff_packet": 500 if QUALITY_REPAIR_MODE else 2500,
    "benchmark_alignment": 1000 if COMPANY_SETUP_REPAIR_MODE else (900 if SOURCE_SELECTION_REPAIR_MODE else (700 if QUALITY_REPAIR_MODE else 3500)),
    "clause_comparison": 600 if QUALITY_REPAIR_MODE else 3000,
    "source_grounded_summary": 350 if QUALITY_REPAIR_MODE else 1200,
    "bilingual_term_mapping": 200 if QUALITY_REPAIR_MODE else 700,
    "clarification": 250 if QUALITY_REPAIR_MODE else 361,
    "refusal": 207,
    "disciplinary_timeline_check": 220 if SOURCE_SELECTION_REPAIR_MODE else 90,
    "epz_applicability": 160 if QUALITY_REPAIR_MODE else 235,
    "foreign_investor_orientation": 120 if QUALITY_REPAIR_MODE else 184,
    "procurement_contract_architecture": 200 if QUALITY_REPAIR_MODE else 300,
}
SMALL_TASK_REPEATS = {
    "refusal": 10 if COMPANY_SETUP_REPAIR_MODE else (8 if QUALITY_REPAIR_MODE else 4),
    "clarification": 4 if QUALITY_REPAIR_MODE else 3,
    "disciplinary_timeline_check": 10 if COMPANY_SETUP_REPAIR_MODE else (14 if SOURCE_SELECTION_REPAIR_MODE else (8 if QUALITY_REPAIR_MODE else 6)),
    "epz_applicability": 3,
    "foreign_investor_orientation": 2,
}

def take_rows(subset, cap, seed):
    if not len(subset) or cap <= 0:
        return None
    shuffled = subset.shuffle(seed=seed)
    return shuffled.select(range(min(cap, len(shuffled))))

def quality_filter_split(split):
    before = len(split)
    filtered = split.filter(quality_row, desc="source-quality filter")
    print("source-quality filter:", before, "->", len(filtered))
    if SOURCE_SELECTION_REPAIR_MODE:
        bad = filtered.filter(lambda row: not source_task_match(row), desc="bad source/task audit")
        assert len(bad) == 0, f"source-selection repair still has {len(bad)} bad source/task rows after filtering"
    return filtered

def build_curriculum(split, max_base_rows=None):
    split = quality_filter_split(split)
    blocks = []
    for offset, (task_type, cap) in enumerate(TASK_TRAIN_CAPS.items()):
        subset = split.filter(lambda row, task_type=task_type: row.get("task_type") == task_type)
        sampled = take_rows(subset, cap, seed=42 + offset)
        if sampled is not None and len(sampled):
            sampled = to_training_columns(sampled)
            repeat = max(1, int(SMALL_TASK_REPEATS.get(task_type, 1)))
            blocks.extend([sampled] * repeat)
    if not blocks:
        raise ValueError("No training rows selected. Check task_type values and quality filters.")
    curriculum = concatenate_datasets(blocks).shuffle(seed=42)
    if max_base_rows and len(curriculum) > max_base_rows:
        curriculum = curriculum.shuffle(seed=43).select(range(max_base_rows))
    return curriculum

company_setup_focus_rows = [row for row in alignment_rows if row.get("task_type") == "company_setup_pathway"]
company_setup_focus_repeat = 260 if COMPANY_SETUP_CITATION_REPAIR_MODE else (160 if COMPANY_SETUP_REPAIR_MODE else 0)
alignment_dataset = Dataset.from_list([
    project_training_row(row)
    for row in (
        alignment_rows * max(1, int(BENCHMARK_ALIGNMENT_REPEAT))
        + company_setup_focus_rows * company_setup_focus_repeat
    )
])
base_target = None
if TARGET_TRAIN_ROWS:
    base_target = max(1, TARGET_TRAIN_ROWS - len(alignment_dataset))

train_base = build_curriculum(raw_dataset["train"], max_base_rows=base_target)
train_split = concatenate_datasets([train_base, alignment_dataset]).shuffle(seed=44)

validation_split = to_training_columns(quality_filter_split(raw_dataset["validation"])).shuffle(seed=42)
if VALIDATION_SAMPLE_ROWS and len(validation_split) > VALIDATION_SAMPLE_ROWS:
    validation_split = validation_split.select(range(VALIDATION_SAMPLE_ROWS))

dataset = DatasetDict({
    "train": train_split,
    "validation": validation_split,
})
print(dataset)
print("training mode:", "company-setup-citation-repair" if COMPANY_SETUP_CITATION_REPAIR_MODE else ("company-setup-repair" if COMPANY_SETUP_REPAIR_MODE else ("source-selection-repair" if SOURCE_SELECTION_REPAIR_MODE else ("quality-repair" if QUALITY_REPAIR_MODE else "full-half-day"))))
print("alignment anchor rows included:", len(alignment_dataset))
print("balanced train tasks:", Counter(dataset["train"]["task_type"]))
print("validation tasks:", Counter(dataset["validation"]["task_type"]))


## 6. Prompt Template

In [ ]:
import json

SYSTEM_PROMPT = (
    "You are a Bangladesh contract, labor, and business-policy vetting "
    "assistant for any business operating in Bangladesh - local SMEs, "
    "family businesses, partnerships, joint ventures, and foreign-invested "
    "companies - across new setup, partnerships, expansion, routine "
    "commercial contracts, and HR. You also handle specialised questions on "
    "EPZ/BEPZA operations, government procurement, and foreign investment. "
    "You help users explore issues before hiring an expert. You are not a "
    "lawyer. Use only the supplied source context. Return exactly one valid "
    "JSON object, with no Markdown, XML tags, or extra prose. Include "
    "citations for source-grounded answers, ask for missing facts when "
    "needed, and refuse unsupported predictions or final legal advice. If "
    "source metadata supplies a citation, source title, or source URL, copy "
    "that metadata into the JSON citations array instead of leaving it empty."
)

def training_context(row):
    context = row.get('context', '') or ''
    if MAX_CONTEXT_CHARS_FOR_TRAINING and len(context) > MAX_CONTEXT_CHARS_FOR_TRAINING:
        return context[:MAX_CONTEXT_CHARS_FOR_TRAINING].rstrip() + "\n[Source excerpt truncated for training; answer only from the visible supplied context.]"
    return context

def prompt_citations(row):
    raw = row.get("citations", [])
    if isinstance(raw, str) and raw.strip():
        try:
            raw = json.loads(raw)
        except Exception:
            raw = []
    citations = normalize_citations(raw) if "normalize_citations" in globals() else []
    if citations:
        return citations
    if row.get("source_title") or row.get("source_url"):
        return [{
            "source_title": row.get("source_title"),
            "source_url": row.get("source_url"),
            "source_type": row.get("source_type"),
            "source_authority": row.get("source_authority"),
            "retrieved_at": row.get("retrieved_at"),
            "section_id": row.get("section_id"),
            "chunk_id": row.get("chunk_id"),
        }]
    return []

def source_metadata(row):
    metadata = {
        "task_type": row.get("task_type", ""),
        "source_title": row.get("source_title", ""),
        "source_url": row.get("source_url", ""),
        "source_type": row.get("source_type", ""),
        "source_authority": row.get("source_authority", ""),
        "section_id": row.get("section_id", ""),
        "chunk_id": row.get("chunk_id", ""),
        "citations_to_copy_when_source_grounded": prompt_citations(row),
    }
    return json.dumps(metadata, ensure_ascii=False)

def render_prompt(row):
    return (
        f"<SYSTEM>{SYSTEM_PROMPT}</SYSTEM>\n"
        f"<INSTRUCTION>{row.get('instruction', '')}</INSTRUCTION>\n"
        f"<SOURCE_METADATA>{source_metadata(row)}</SOURCE_METADATA>\n"
        f"<CONTEXT>{training_context(row)}</CONTEXT>\n"
        f"<RESPONSE>"
    )

def render_full(row):
    response = (row.get("response") or "").strip()
    if RESPONSE_ONLY_TRAINING:
        return render_prompt(row) + response + "</RESPONSE>"
    reasoning = (row.get("reasoning") or "").strip()
    if reasoning:
        return render_prompt(row) + f"<REASONING>{reasoning}</REASONING>\n<FINAL>{response}</FINAL></RESPONSE>"
    return render_prompt(row) + response + "</RESPONSE>"

print(render_full(dataset["train"][0])[:1200])


## 7. Tokenizer And Model

In [ ]:
import torch
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

if "SKIP_TRAINING" not in globals():
    SKIP_TRAINING = False

# Company-setup repair trains from the source-selection repair adapter.
# If SKIP_TRAINING=True, load the current output adapter for a no-cost smoke test.
if SKIP_TRAINING:
    ADAPTER_TO_LOAD = HF_OUTPUT_REPO
    ADAPTER_IS_TRAINABLE = False
elif QUALITY_REPAIR_MODE:
    ADAPTER_TO_LOAD = BASE_ADAPTER_REPO
    ADAPTER_IS_TRAINABLE = True
else:
    ADAPTER_TO_LOAD = None
    ADAPTER_IS_TRAINABLE = True

print("adapter_to_load:", ADAPTER_TO_LOAD or "(new LoRA)")
print("adapter_is_trainable:", ADAPTER_IS_TRAINABLE)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True, token=os.environ["HF_TOKEN"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16,
    token=os.environ["HF_TOKEN"],
)
model.config.use_cache = bool(SKIP_TRAINING)

if not SKIP_TRAINING:
    prep_kwargs = {"use_gradient_checkpointing": USE_GRADIENT_CHECKPOINTING}
    if USE_GRADIENT_CHECKPOINTING:
        prep_kwargs["gradient_checkpointing_kwargs"] = {"use_reentrant": False}
    model = prepare_model_for_kbit_training(model, **prep_kwargs)
    if USE_GRADIENT_CHECKPOINTING:
        model.enable_input_require_grads()

if ADAPTER_TO_LOAD:
    try:
        print("Loading adapter:", ADAPTER_TO_LOAD)
        model = PeftModel.from_pretrained(
            model,
            ADAPTER_TO_LOAD,
            is_trainable=ADAPTER_IS_TRAINABLE,
            token=os.environ["HF_TOKEN"],
        )
    except Exception as exc:
        if SKIP_TRAINING and QUALITY_REPAIR_MODE and ADAPTER_TO_LOAD != BASE_ADAPTER_REPO:
            print("Could not load repaired adapter; falling back to base repair adapter for smoke testing.")
            print(type(exc).__name__, exc)
            model = PeftModel.from_pretrained(
                model,
                BASE_ADAPTER_REPO,
                is_trainable=False,
                token=os.environ["HF_TOKEN"],
            )
        else:
            raise
else:
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
    lora_config = LoraConfig(
        r=32,
        lora_alpha=64,
        lora_dropout=0.03,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=target_modules,
    )
    model = get_peft_model(model, lora_config)

if hasattr(model, "print_trainable_parameters"):
    model.print_trainable_parameters()
model.eval() if SKIP_TRAINING else model.train()


## 8. Tokenize With Response-Only Loss

In [ ]:
if SKIP_TRAINING:
    tokenized = None
    print("SKIP_TRAINING=True; tokenization skipped. Run All will continue to sanity inference without paid training.")
else:
    def tokenize_row(row):
        prompt = render_prompt(row)
        full = render_full(row)
        encoded = tokenizer(full, truncation=True, max_length=MAX_LEN, add_special_tokens=True)
        prompt_encoded = tokenizer(prompt, truncation=True, max_length=MAX_LEN, add_special_tokens=True)
        labels = list(encoded["input_ids"])
        prompt_len = min(len(prompt_encoded["input_ids"]), len(labels))
        for i in range(prompt_len):
            labels[i] = -100
        if labels and all(label == -100 for label in labels):
            labels[-1] = encoded["input_ids"][-1]
        encoded["labels"] = labels
        return encoded

    tokenized = dataset.map(tokenize_row, remove_columns=dataset["train"].column_names, desc="tokenize vetting rows")
    print(tokenized)


## 9. Train

In [ ]:
import inspect, os, time, traceback
from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments, TrainerCallback
from transformers.trainer_utils import get_last_checkpoint
from huggingface_hub import HfApi, create_repo, upload_folder

api = HfApi(token=os.environ["HF_TOKEN"])

def verify_adapter_dir(path):
    files = set(os.listdir(path)) if os.path.isdir(path) else set()
    assert "adapter_config.json" in files, f"adapter_config.json missing in {path}"
    assert ("adapter_model.safetensors" in files) or ("adapter_model.bin" in files), f"adapter weights missing in {path}"
    return files

def save_adapter_copy(model, target_dir, label):
    os.makedirs(target_dir, exist_ok=True)
    model.save_pretrained(target_dir, safe_serialization=True)
    tokenizer.save_pretrained(target_dir)
    files = verify_adapter_dir(target_dir)
    manifest_path = os.path.join(target_dir, "persistence_manifest.json")
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump({
            "label": label,
            "run_id": RUN_ID,
            "run_name": RUN_NAME,
            "dataset_repo": DATA_REPO,
            "adapter_repo": HF_OUTPUT_REPO,
            "saved_at": time.time(),
        }, f, indent=2)
    print(f"{label} saved:", target_dir, sorted(files))
    return files

def verify_remote_adapter(path_in_repo=""):
    prefix = (path_in_repo.strip("/") + "/") if path_in_repo else ""
    remote_files = set(api.list_repo_files(HF_OUTPUT_REPO, repo_type="model"))
    assert prefix + "adapter_config.json" in remote_files, f"remote adapter_config.json missing at {prefix or 'repo root'}"
    assert (prefix + "adapter_model.safetensors" in remote_files) or (prefix + "adapter_model.bin" in remote_files), f"remote adapter weights missing at {prefix or 'repo root'}"
    return remote_files

def upload_adapter_copy(source_dir, path_in_repo, label):
    verify_adapter_dir(source_dir)
    create_repo(HF_OUTPUT_REPO, repo_type="model", private=True, exist_ok=True, token=os.environ["HF_TOKEN"])
    print("uploading adapter folder:", source_dir)
    print("to:", "https://huggingface.co/" + HF_OUTPUT_REPO + "/tree/main/" + path_in_repo)
    for attempt in range(1, 5):
        try:
            upload_folder(
                repo_id=HF_OUTPUT_REPO,
                repo_type="model",
                folder_path=source_dir,
                path_in_repo=path_in_repo,
                token=os.environ["HF_TOKEN"],
                commit_message=f"{label} attempt {attempt}",
            )
            break
        except Exception as exc:
            print(f"upload attempt {attempt} failed:", type(exc).__name__, exc)
            traceback.print_exc()
            time.sleep(5 * attempt)
    else:
        raise RuntimeError("all upload attempts failed for " + path_in_repo)
    verify_remote_adapter(path_in_repo)
    print(f"{label} verified on Hub at {path_in_repo}")

class AdapterPersistenceCallback(TrainerCallback):
    def on_save(self, args, state, control, model=None, **kwargs):
        if model is None:
            raise RuntimeError("Trainer did not provide model to persistence callback")
        if state.global_step <= 0:
            return control
        checkpoint_name = f"checkpoint-{state.global_step}"
        drive_checkpoint_dir = os.path.join(DRIVE_BACKUP_DIR, checkpoint_name)
        if BACKUP_TO_DRIVE:
            save_adapter_copy(model, drive_checkpoint_dir, f"Drive adapter checkpoint step {state.global_step}")
        if HUB_BACKUP_EVERY_SAVE:
            source_dir = drive_checkpoint_dir if BACKUP_TO_DRIVE else os.path.join(LOCAL_WORK_DIR, checkpoint_name)
            if not os.path.isdir(source_dir):
                save_adapter_copy(model, source_dir, f"Local adapter checkpoint step {state.global_step}")
            upload_adapter_copy(
                source_dir,
                f"{HUB_CHECKPOINT_PREFIX}/{checkpoint_name}",
                f"Verified adapter-checkpoints/{checkpoint_name}",
            )
        return control

TRAINING_WAS_SKIPPED = bool(SKIP_TRAINING)
trainer = None

if SKIP_TRAINING:
    print("SKIP_TRAINING=True; skipping Trainer setup, paid training, checkpoints, and Hub checkpoint uploads.")
else:
    resume_from = None
    if RESUME_FROM_DRIVE_CHECKPOINT and os.path.isdir(OUTPUT_DIR):
        latest_checkpoint = get_last_checkpoint(OUTPUT_DIR)
        if latest_checkpoint:
            resume_from = latest_checkpoint
            print("will resume from Drive Trainer checkpoint:", resume_from)
        else:
            print("no Drive Trainer checkpoint found; starting from base adapter init")
    else:
        print("Drive checkpoint resume disabled or output dir missing; starting from base adapter init")

    collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        pad_to_multiple_of=8,
        label_pad_token_id=-100,
        return_tensors="pt",
    )
    effective_batch = max(1, BATCH_SIZE * GRAD_ACCUM)
    estimated_total_steps = MAX_STEPS if MAX_STEPS and MAX_STEPS > 0 else max(1, int((len(tokenized["train"]) * NUM_EPOCHS + effective_batch - 1) // effective_batch))
    WARMUP_STEPS = min(100, max(20, int(estimated_total_steps * 0.03)))
    if QUALITY_REPAIR_MODE:
        EVAL_SAVE_STEPS = max(100, min(250, max(1, estimated_total_steps // 3)))
    else:
        EVAL_SAVE_STEPS = max(500, min(1000, max(1, estimated_total_steps // 4)))
    print({"estimated_total_steps": estimated_total_steps, "warmup_steps": WARMUP_STEPS, "eval_save_steps": EVAL_SAVE_STEPS})

    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        overwrite_output_dir=True,
        num_train_epochs=NUM_EPOCHS,
        max_steps=MAX_STEPS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=GRAD_ACCUM,
        gradient_checkpointing=USE_GRADIENT_CHECKPOINTING,
        gradient_checkpointing_kwargs={"use_reentrant": False} if USE_GRADIENT_CHECKPOINTING else None,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_steps=WARMUP_STEPS,
        logging_steps=25,
        eval_strategy="steps",
        eval_steps=EVAL_SAVE_STEPS,
        save_strategy="steps",
        save_steps=EVAL_SAVE_STEPS,
        save_total_limit=2,
        optim="paged_adamw_8bit",
        max_grad_norm=0.3,
        bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
        fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
        report_to="tensorboard",
        run_name=RUN_NAME,
        remove_unused_columns=False,
        group_by_length=True,
        dataloader_num_workers=2,
    )

    trainer_kwargs = dict(
        model=model,
        args=training_args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["validation"],
        data_collator=collator,
        callbacks=[AdapterPersistenceCallback()],
    )
    if "processing_class" in inspect.signature(Trainer.__init__).parameters:
        trainer_kwargs["processing_class"] = tokenizer
    else:
        trainer_kwargs["tokenizer"] = tokenizer

    trainer = Trainer(**trainer_kwargs)
    trainer.train(resume_from_checkpoint=resume_from)
    TRAINING_WAS_SKIPPED = False


## 10. Evaluate, Save, And Push

In [ ]:
import json, math, os
from huggingface_hub import create_repo, upload_file

if SKIP_TRAINING or globals().get("trainer") is None:
    metrics = {
        "training_skipped": True,
        "base_model": BASE_MODEL,
        "dataset_repo": DATA_REPO,
        "adapter_repo": HF_OUTPUT_REPO,
        "loaded_adapter_repo": ADAPTER_TO_LOAD,
        "note": "Evaluate/Save/Push skipped because SKIP_TRAINING=True. Sanity inference will run on the loaded adapter.",
    }
    print(json.dumps(metrics, indent=2))
else:
    metrics = trainer.evaluate()
    if "eval_loss" in metrics:
        metrics["eval_perplexity"] = float(math.exp(metrics["eval_loss"])) if metrics["eval_loss"] < 20 else None
    metrics["base_model"] = BASE_MODEL
    metrics["dataset_repo"] = DATA_REPO
    metrics["adapter_repo"] = HF_OUTPUT_REPO
    metrics["drive_backup_dir"] = DRIVE_BACKUP_DIR
    metrics["final_drive_adapter_dir"] = DRIVE_FINAL_ADAPTER_DIR
    metrics["trainer_output_dir"] = OUTPUT_DIR
    print(json.dumps(metrics, indent=2))

    create_repo(HF_OUTPUT_REPO, repo_type="model", private=True, exist_ok=True, token=os.environ["HF_TOKEN"])

    save_adapter_copy(trainer.model, FINAL_ADAPTER_DIR, "Final local adapter")
    if BACKUP_TO_DRIVE:
        save_adapter_copy(trainer.model, DRIVE_FINAL_ADAPTER_DIR, "Final Drive adapter")

    for report_dir in [FINAL_ADAPTER_DIR, DRIVE_FINAL_ADAPTER_DIR if BACKUP_TO_DRIVE else None, OUTPUT_DIR]:
        if report_dir:
            os.makedirs(report_dir, exist_ok=True)
            with open(os.path.join(report_dir, "eval_report.json"), "w", encoding="utf-8") as f:
                json.dump(metrics, f, indent=2)

    # Root push keeps standard PEFT loading simple; final-adapter/ is a verified snapshot.
    trainer.model.push_to_hub(HF_OUTPUT_REPO, private=True, token=os.environ["HF_TOKEN"], commit_message="Upload final Bangladesh contract labor policy vetting LoRA")
    tokenizer.push_to_hub(HF_OUTPUT_REPO, private=True, token=os.environ["HF_TOKEN"], commit_message="Upload tokenizer")
    upload_file(
        path_or_fileobj=os.path.join(FINAL_ADAPTER_DIR, "eval_report.json"),
        path_in_repo="eval_report.json",
        repo_id=HF_OUTPUT_REPO,
        repo_type="model",
        token=os.environ["HF_TOKEN"],
        commit_message="Upload eval report",
    )
    upload_adapter_copy(FINAL_ADAPTER_DIR, FINAL_HUB_SUBFOLDER, "Final adapter snapshot")
    print("Pushed final adapter to", HF_OUTPUT_REPO)
    print("Verified final adapter on Drive:", DRIVE_FINAL_ADAPTER_DIR)
    print("Verified final adapter on Hub:", "https://huggingface.co/" + HF_OUTPUT_REPO + "/tree/main/" + FINAL_HUB_SUBFOLDER)


## 11. Sanity Inference

In [ ]:
import copy, json, re, torch
from transformers import GenerationConfig


def first_balanced_json(text):
    start = text.find("{")
    if start < 0:
        return text.strip()
    depth = 0
    in_string = False
    escape = False
    for idx, ch in enumerate(text[start:], start=start):
        if escape:
            escape = False
            continue
        if ch == "\\":
            escape = True
            continue
        if ch == '"':
            in_string = not in_string
            continue
        if in_string:
            continue
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return text[start:idx + 1].strip()
    return text[start:].strip()


def clean_generation(text):
    for marker in ("</RESPONSE>", "<SYSTEM>", "<INSTRUCTION>", "<CONTEXT>"):
        if marker in text:
            text = text.split(marker, 1)[0]
    if "<FINAL>" in text:
        text = text.split("<FINAL>", 1)[-1].split("</FINAL>", 1)[0].strip()
    return first_balanced_json(text.strip())


def deterministic_generation_config(max_new_tokens=1100):
    config = GenerationConfig.from_model_config(model.config)
    config.do_sample = False
    config.num_beams = 1
    config.repetition_penalty = 1.03
    config.max_new_tokens = max_new_tokens
    config.pad_token_id = tokenizer.pad_token_id
    config.eos_token_id = tokenizer.eos_token_id
    # Qwen chat configs often carry sampling defaults. Clear them for greedy
    # benchmark runs so Transformers does not warn that they are ignored.
    for attr in ("temperature", "top_p", "top_k", "min_p", "typical_p"):
        if hasattr(config, attr):
            setattr(config, attr, None)
    return config


model.generation_config = deterministic_generation_config(max_new_tokens=1100)


def generate_for_row(row, max_new_tokens=1100, json_prefill=True):
    prompt = render_prompt(row)
    prefix = "{\n" if json_prefill else ""
    device = next(model.parameters()).device
    inputs = tokenizer(prompt + prefix, return_tensors="pt", truncation=True, max_length=MAX_LEN).to(device)
    gen_config = deterministic_generation_config(max_new_tokens=max_new_tokens)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            generation_config=gen_config,
        )
    raw = tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    return clean_generation(prefix + raw)


def safe_fallback_json(row, generated):
    """Make the smoke cell non-destructive when a probe fails JSON.

    This fallback is only for notebook diagnostics. It is marked explicitly and
    is not used for training or saving the model.
    """
    task = row.get("task_type", "unknown")
    citations = normalize_citations(row.get("citations", [])) if "normalize_citations" in globals() else []
    payload = {
        "risk_level": "review_required",
        "task_type": task,
        "diagnostic_fallback_used": True,
        "issue": "The model did not emit valid JSON for this smoke-test probe.",
        "safe_next_step": "Continue or repeat the repair pass, and use source-grounded retrieval plus JSON validation in production.",
        "raw_generation_preview": generated[:500],
        "citations": citations,
        "disclaimer": ALIGNMENT_DISCLAIMER if "ALIGNMENT_DISCLAIMER" in globals() else "Automated exploration support, not legal advice.",
    }
    if task == "disciplinary_timeline_check":
        payload.update({
            "status": "do_not_proceed_without_expert_review",
            "missing_facts_to_confirm": ["worker category", "EPZ/non-EPZ regime", "notice", "response", "inquiry record", "payments"],
        })
    if task == "company_setup_pathway":
        payload.update({
            "source_limit": "Use only Companies Act/RJSC-relevant excerpts for incorporation and setup checkpoints.",
            "missing_facts_to_confirm": ["entity type", "shareholders/directors", "capital", "address", "sector", "foreign ownership"],
        })
    return json.dumps(payload, ensure_ascii=False, indent=2)


model.eval()
probe_rows = alignment_rows
score = {
    "valid_json_raw": 0,
    "valid_json_after_fallback": 0,
    "fallback_used": 0,
    "no_training_tags": 0,
    "has_disclaimer_or_refusal": 0,
    "has_citations_or_refusal": 0,
    "bad_source_task_matches": 0,
    "weak_company_setup_sources": 0,
    "missing_company_setup_citations": 0,
    "total": len(probe_rows),
}
failed_probes = []
bad_source_probes = []
weak_company_setup_probes = []
missing_company_setup_citation_probes = []
for idx, probe in enumerate(probe_rows, start=1):
    generated = generate_for_row(probe)
    has_tags = any(tag in generated for tag in ("<REASONING>", "<FINAL>", "<SYSTEM>", "<INSTRUCTION>", "<CONTEXT>"))
    score["no_training_tags"] += int(not has_tags)
    try:
        parsed = json.loads(generated)
        score["valid_json_raw"] += 1
    except Exception:
        failed_probes.append({"idx": idx, "task_type": probe.get("task_type"), "preview": generated[:500]})
        generated = safe_fallback_json(probe, generated)
        parsed = json.loads(generated)
        score["fallback_used"] += 1
    score["valid_json_after_fallback"] += 1
    lowered = json.dumps(parsed, ensure_ascii=False).lower()
    score["has_disclaimer_or_refusal"] += int("not legal advice" in lowered or "cannot" in lowered or "do_not_proceed" in lowered)
    score["has_citations_or_refusal"] += int(bool(parsed.get("citations")) or "cannot" in lowered or "do_not_proceed" in lowered)
    if not source_task_match(probe):
        score["bad_source_task_matches"] += 1
        bad_source_probes.append({
            "idx": idx,
            "task_type": probe.get("task_type"),
            "source_title": probe.get("source_title"),
            "source_url": probe.get("source_url"),
        })
    if weak_company_setup_source(probe):
        score["weak_company_setup_sources"] += 1
        weak_company_setup_probes.append({
            "idx": idx,
            "task_type": probe.get("task_type"),
            "source_title": probe.get("source_title"),
            "source_url": probe.get("source_url"),
            "source_grounding_preview": str(parsed.get("source_grounding", probe.get("context", "")))[:500],
        })
    if probe.get("task_type") == "company_setup_pathway" and not parsed.get("citations"):
        score["missing_company_setup_citations"] += 1
        missing_company_setup_citation_probes.append({
            "idx": idx,
            "task_type": probe.get("task_type"),
            "source_grounding_preview": str(parsed.get("source_grounding", ""))[:500],
        })
    print("\n=== probe", idx, probe.get("task_type"), "===")
    print(generated[:2600])
print("\nbehavior_smoke_score", score)
if failed_probes:
    print("\nSmoke-test probes needing more repair training or runtime JSON validation:")
    print(json.dumps(failed_probes, indent=2, ensure_ascii=False))
else:
    print("\nSmoke test passed JSON/fallback audit.")
if bad_source_probes:
    print("\nSmoke-test probes with bad source/task matches:")
    print(json.dumps(bad_source_probes, indent=2, ensure_ascii=False))
else:
    print("Smoke test passed source/task matching audit.")
if weak_company_setup_probes:
    print("\nSmoke-test company setup probes still using weak setup sources:")
    print(json.dumps(weak_company_setup_probes, indent=2, ensure_ascii=False))
else:
    print("Smoke test passed company setup source-strength audit.")
if missing_company_setup_citation_probes:
    print("\nSmoke-test company setup probes missing citations:")
    print(json.dumps(missing_company_setup_citation_probes, indent=2, ensure_ascii=False))
else:
    print("Smoke test passed company setup citation audit.")
if not failed_probes and not bad_source_probes and not weak_company_setup_probes and not missing_company_setup_citation_probes:
    print("Smoke test passed without fallback.")
